# Fase 2 — Do Protótipo ao Agente: Ferramentas, Raciocínio ReAct e Prompts Avançados

A chain da Fase 1 transforma texto em JSON, mas decide tudo de antemão. Nesta fase o MVP vira um **agente**: decide, a cada passo, se responde ou se usa uma ferramenta — incluindo **APIs externas reais** (PyPI, OSV.dev, BrasilAPI e GitHub).

Conteúdo: comparação de orquestradores · ReAct clássico com `initialize_agent` · *function calling* com APIs externas · chains sequenciais, paralelas e `RouterChain` · prompt PTCF com *few-shot* e *chain-of-thought* · confiabilidade (parsers com tratamento de erro, memória, FAISS e *callbacks*).

In [ ]:
import sys, pathlib, json, time, warnings
RAIZ = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "fabrica_agentica").is_dir())
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", message=".*LangChainDeprecationWarning.*")

from fabrica_agentica.config import descrever_ambiente, criar_llm, texto_de
print(descrever_ambiente())

Provedor de LLM : anthropic
APIs externas   : rede (ao vivo)
Embeddings      : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Modelo NLI      : MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli
LangSmith       : ligado
Repositório GH  : DriccoC/fabrica-agentica-sandbox


## 1. Comparação de orquestradores de agentes

Critérios objetivos: **autonomia** (quanto o framework deixa o agente decidir sozinho o próximo passo), **abstração** (quanto esconde do desenvolvedor) e **modularidade** (quanto se consegue trocar peças: modelo, ferramentas, memória, *vector store*).

| Critério | LangChain (+ LangGraph) | LlamaIndex | CrewAI | AutoGen (Microsoft) |
|---|---|---|---|---|
| Autonomia | de chain fixa a agente ReAct e grafos com ciclos; o desenvolvedor escolhe o grau | alta para RAG (agentes de consulta); menor para ações gerais | alta, mas orientada a "tripulações" com papéis | alta, baseada em conversa entre agentes |
| Abstração | média: construtos prontos (`AgentExecutor`, `create_agent`) com acesso a cada peça | alta para indexação/consulta; esconde o pipeline | alta: papéis, tarefas e processos declarativos | média: troca de mensagens entre agentes |
| Modularidade | muito alta: centenas de integrações de modelos, ferramentas, memórias e *vector stores* com a mesma interface `Runnable` | alta em dados (loaders, índices); menor em ferramentas | média: depende de integrações de terceiros (muitas do próprio LangChain) | média |
| Agente único com ferramentas e memória | nativo | possível | pensado para múltiplos agentes | pensado para múltiplos agentes |
| Observabilidade e avaliação | LangSmith integrado (traces, datasets, avaliadores) | integrações de terceiros | integrações de terceiros | integrações de terceiros |
| Caminho para multiagente | LangGraph (mesmo ecossistema) | Workflows | nativo | nativo |

**Por que LangChain para continuar o MVP:** (1) o MVP já é LCEL, e os agentes do LangChain consomem os mesmos prompts, parsers e ferramentas, sem reescrita; (2) o projeto exige agente **único** agora e multiagente na próxima disciplina — o LangGraph é a evolução natural dentro do mesmo ecossistema, enquanto CrewAI e AutoGen já partem de múltiplos agentes; (3) a avaliação exigida (qualidade e trajetória) é feita no LangSmith, que é nativo; (4) a modularidade permite trocar o modelo (Claude, OpenAI ou o modelo de demonstração) sem mudar o agente. LlamaIndex seria forte se o problema fosse só RAG, mas o agente precisa sobretudo de ferramentas de ação e cálculo.

## 2. Agente ReAct clássico: `initialize_agent` + `ZERO_SHOT_REACT_DESCRIPTION`

O ReAct alterna, em texto, **Thought → Action → Action Input → Observation** até chegar à **Final Answer**. O tipo `ZERO_SHOT_REACT_DESCRIPTION` escolhe a ferramenta apenas pela descrição (zero-shot) e aceita ferramentas de uma entrada (string). Memória básica: `ConversationBufferMemory`, injetada como `{chat_history}`.

In [2]:
from fabrica_agentica.agentes.react_classico import criar_agente_react

agente_react = criar_agente_react(verbose=True)
print(agente_react.agent.llm_chain.prompt.template[:1400], "\n...")

/Users/adrianocarvalho/Documents/_MBA/Agentes_Cognitivos/projeto_fabrica_agentica/fabrica_agentica/agentes/react_classico.py:76: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memoria = ConversationBufferMemory(memory_key="chat_history", input_key="input")
/Users/adrianocarvalho/Documents/_MBA/Agentes_Cognitivos/projeto_fabrica_agentica/.venv/lib/python3.12/site-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.

Você é o Tech Lead Agêntico da NovaForja Software (empresa fictícia). Responda em português do Brasil. Nunca invente números, versões ou regras: use as ferramentas. Você tem acesso às seguintes ferramentas:

calcular_capacidade_sprint(projeto) - Capacidade e saldo de pontos da próxima sprint planejada. Entrada: nome do projeto (Portal, PagaFacil ou AppCampo).
consultar_pypi(pacote) - Versão estável mais recente de um pacote Python no PyPI. Entrada: nome do pacote.
consultar_vulnerabilidades(texto: 'str') -> 'str' - Vulnerabilidades conhecidas (OSV) de uma versão de pacote. Entrada no formato pacote==versao (ex.: authlib==1.3.0).
buscar_manual_engenharia(consulta) - Busca no Manual de Engenharia (normas MAN-01 a MAN-07). Entrada: consulta com termos formais das normas.
consultar_backlog_sql(consulta) - Executa um SELECT no banco de backlog. Entrada: uma consulta SQL SQLite. Banco de backlog da NovaForja Software (SQLite). Tabelas e colunas: - projetos(projeto_id, nome, descricao, stack,

/Users/adrianocarvalho/Documents/_MBA/Agentes_Cognitivos/projeto_fabrica_agentica/fabrica_agentica/agentes/react_classico.py:77: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  return initialize_agent(


In [3]:
r1 = agente_react.invoke({"input": "A biblioteca pyjwt na versão 2.8.0 é segura para usarmos?"})
print("\nRESPOSTA:", r1["output"])



> Entering new AgentExecutor chain...


Preciso verificar as vulnerabilidades conhecidas dessa versão do pacote e também consultar a norma interna aplicável.

Action: consultar_vulnerabilidades
Action Input: pyjwt==2.8.0



Observation: {"pacote": "pyjwt", "versao": "2.8.0", "total": 28, "vulnerabilidades": [{"id": "GHSA-ffc3-869f-jxw9", "aliases": ["CVE-2026-102268", "PYSEC-2026-4145"], "resumo": "PyJWT: Asymmetric-PEM detection bypass: whitespace/line-ending-mutated public keys skip the HS/asymmetric confusion guard", "cvss": 9.1, "severidade_rotulo": "CRITICAL", "corrigida_em": ["2.14.0"]}, {"id": "PYSEC-2026-4145", "aliases": ["CVE-2026-102268", "GHSA-ffc3-869f-jxw9"], "resumo": "PyJWT: Asymmetric-PEM detection bypass: whitespace/line-ending-mutated public keys skip the HS/asymmetric confusion guard", "cvss": 9.1, "severidade_rotulo": null, "corrigida_em": ["2.14.0"]}, {"id": "PYSEC-2026-4183", "aliases": ["CVE-2026-102275", "GHSA-x33g-cr3x-6449"], "resumo": "", "cvss": 8.2, "severidade_rotulo": null, "corrigida_em": ["2.15.0"]}, {"id": "GHSA-752w-5fwx-jx9f", "aliases": ["CVE-2026-32597", "PYSEC-2026-120"], "resumo": "PyJWT accepts unknown `crit` header extensions", "cvss": 7.5, "severidade_rotulo": 

Thought: Encontrei vulnerabilidades críticas. Agora preciso consultar a norma interna aplicável para saber a regra de dependências vulneráveis, e também verificar a versão estável mais recente.

Action: buscar_manual_engenharia
Action Input: dependências vulneráveis CVSS severidade crítica prazo de correção bibliotecas de terceiros

W1105 17:08:55.846000 70831 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Observation: {"consulta": "dependências vulneráveis CVSS severidade crítica prazo de correção bibliotecas de terceiros", "aviso": null, "resultados": [{"chunk_id": "CHK_019", "doc_id": "MAN-04", "secao": "2. Critérios de admissão de componentes", "distancia_l2": 0.633, "similaridade": "alta", "trecho": "## 2. Critérios de admissão de componentes\n1. É vedada a adoção de componente com vulnerabilidade conhecida de pontuação CVSS igual ou superior a 7,0 (severidade alta ou crítica) na versão pretendida, salvo exceção formal aprovada pelo time de segurança com prazo de correção.\n2. Vulnerabilidades de severidade média (CVSS de 4,0 a 6,9) devem ser registradas no backlog com prazo de correção de até 30 dias.\n3. A versão adotada deve ser a mais recente estável disponível no repositório oficial do ecossistema, ou estar no máximo uma versão secundária (MINOR) atrás dela."}, {"chunk_id": "CHK_003", "doc_id": "MAN-01", "secao": "2. Critérios obrigatórios de conclusão", "distancia_l2": 0.938,

Thought: Já tenho as vulnerabilidades e a norma MAN-04. Falta verificar a versão estável mais recente no PyPI para avaliar o critério de versão.

Action: consultar_pypi
Action Input: pyjwt
Observation: {"pacote": "pyjwt", "encontrado": true, "versao_mais_recente": "2.15.1", "resumo": "JSON Web Token implementation in Python", "ultimas_versoes": [{"versao": "2.15.1", "data": "2026-09-28"}, {"versao": "2.15.0", "data": "2026-09-23"}, {"versao": "2.14.0", "data": "2026-09-11"}, {"versao": "2.13.0", "data": "2026-05-21"}, {"versao": "2.12.1", "data": "2026-03-13"}]}
Thought:

Thought: I now know the final answer

Final Answer: **Não, a pyjwt 2.8.0 não é segura e não pode ser adotada** segundo a norma MAN-04.

**Vulnerabilidades (OSV)**
A consulta retornou 28 registros para essa versão, e o maior CVSS é 9,1. Os principais, com as correções:
- **GHSA-ffc3-869f-jxw9 / CVE-2026-102268 (CVSS 9,1, crítica):** um bypass na detecção de chave PEM assimétrica. Chaves públicas com espaços ou quebras de linha alteradas escapam da proteção contra confusão entre HS e algoritmos assimétricos. Corrigida na 2.14.0.
- **PYSEC-2026-4183 / CVE-2026-102275 (CVSS 8,2, alta):** corrigida na 2.15.0. O registro não traz resumo.
- **GHSA-752w-5fwx-jx9f / CVE-2026-32597 (CVSS 7,5, alta):** a biblioteca aceita extensões desconhecidas no cabeçalho `crit`. Corrigida na 2.12.0.

Mostrei apenas os registros retornados pela consulta, que lista cinco entradas. Alguns são aliases do mesmo problema, e o total de 28 não foi detalhado por completo.

**O que diz o Manual de Engenharia**
- **MAN-

In [4]:
# Segundo turno: "dela" só faz sentido com a memória do turno anterior
r2 = agente_react.invoke({"input": "E qual é a versão mais recente dela?"})
print("\nRESPOSTA:", r2["output"])
print("\nMEMÓRIA (ConversationBufferMemory):\n", agente_react.memory.buffer[:600])



> Entering new AgentExecutor chain...


Thought: O usuário pergunta a versão mais recente da pyjwt. Embora o histórico mencione a 2.15.1, devo confirmar com a ferramenta, em vez de depender apenas da conversa anterior.

Action: consultar_pypi
Action Input: pyjwt
Observation: {"pacote": "pyjwt", "encontrado": true, "versao_mais_recente": "2.15.1", "resumo": "JSON Web Token implementation in Python", "ultimas_versoes": [{"versao": "2.15.1", "data": "2026-09-28"}, {"versao": "2.15.0", "data": "2026-09-23"}, {"versao": "2.14.0", "data": "2026-09-11"}, {"versao": "2.13.0", "data": "2026-05-21"}, {"versao": "2.12.1", "data": "2026-03-13"}]}
Thought:

Thought: A consulta ao PyPI confirmou a versão. Já tenho a resposta, e não preciso de outra ferramenta para esta pergunta.

Final Answer: A versão estável mais recente da **pyjwt** no PyPI é a **2.15.1**, publicada em **28/09/2026**. Isso confirma o que eu havia informado antes.

Versões recentes retornadas pela consulta:
- 2.15.1 (28/09/2026)
- 2.15.0 (23/09/2026)
- 2.14.0 (11/09/2026)
- 2.13.0 (21/05/2026)
- 2.12.1 (13/03/2026)

Ainda não consultei as vulnerabilidades da 2.15.1, então não posso garantir que ela esteja livre de problemas. Pela MAN-04, antes de adotá-la é preciso rodar essa verificação e fixar a versão no arquivo de dependências. Quer que eu consulte `pyjwt==2.15.1` agora?

> Finished chain.

RESPOSTA: A versão estável mais recente da **pyjwt** no PyPI é a **2.15.1**, publicada em **28/09/2026**. Isso confirma o que eu havia informado antes.

Versões recentes retornadas pela consulta:
- 2.15.1 (28/09/2026)
- 2.15.0 (23/09/2026)
- 2.14.0 (11/09/2026)
- 2.13.0 (21/05/202

## 3. API externa via *function calling*

Com *function calling*, o modelo não escreve "Action: ..." em texto: ele devolve uma **chamada estruturada** (`tool_calls`, com nome e argumentos em JSON) e o código executa a função. Abaixo, o ciclo sem o executor, para ver cada etapa: o Claude decide chamar a API do PyPI, executamos a chamada real e devolvemos o resultado como `ToolMessage`.

In [5]:
from langchain_core.messages import HumanMessage, ToolMessage
from fabrica_agentica.ferramentas.catalogo import consultar_pypi, consultar_vulnerabilidades, consultar_feriados

ferramentas_api = {t.name: t for t in (consultar_pypi, consultar_vulnerabilidades, consultar_feriados)}
llm_com_ferramentas = criar_llm("agente").bind_tools(list(ferramentas_api.values()))

mensagens = [HumanMessage("Quais feriados nacionais caem em novembro de 2026 e qual a versão mais recente do fastapi?")]
resposta = llm_com_ferramentas.invoke(mensagens)
print("Chamadas decididas pelo modelo:", json.dumps(resposta.tool_calls, ensure_ascii=False, indent=1))

mensagens.append(resposta)
for chamada in resposta.tool_calls:
    saida = ferramentas_api[chamada["name"]].invoke(chamada["args"])       # chamada HTTP real
    print(f"\n<- {chamada['name']}: {saida[:200]}...")
    mensagens.append(ToolMessage(content=saida, tool_call_id=chamada["id"], name=chamada["name"]))

print("\nRESPOSTA FINAL:\n", texto_de(llm_com_ferramentas.invoke(mensagens)))

Chamadas decididas pelo modelo: [
 {
  "name": "consultar_feriados",
  "args": {
   "ano": 2026
  },
  "id": "toolu_01R8uD9huarWrqDFYqSk9A9r",
  "type": "tool_call"
 },
 {
  "name": "consultar_pypi",
  "args": {
   "pacote": "fastapi"
  },
  "id": "toolu_01VRmGNPVCgtwSyhARYUCYrp",
  "type": "tool_call"
 }
]



<- consultar_feriados: [{"date": "2026-01-01", "name": "Confraternização mundial"}, {"date": "2026-02-16", "name": "Carnaval"}, {"date": "2026-02-17", "name": "Carnaval"}, {"date": "2026-04-03", "name": "Sexta-feira Santa"}...

<- consultar_pypi: {"pacote": "fastapi", "encontrado": true, "versao_mais_recente": "0.162.0", "resumo": "FastAPI framework, high performance, easy to learn, fast to code, ready for production", "ultimas_versoes": [{"ve...



RESPOSTA FINAL:
 **Feriados nacionais em novembro de 2026:**
- **2 de novembro (segunda-feira):** Finados
- **15 de novembro (domingo):** Proclamação da República
- **20 de novembro (sexta-feira):** Dia da Consciência Negra

**FastAPI:** a versão estável mais recente no PyPI é a **0.162.0**, publicada em 23/10/2026.


## 4. Chains sequenciais e paralelas

- **Sequencial:** demanda → resumo → histórias → riscos; cada etapa recebe a saída das anteriores (`RunnablePassthrough.assign`).
- **Paralela:** três checagens independentes ao mesmo tempo — normas (FAISS), biblioteca (PyPI + OSV) e capacidade (SQL + feriados) — com `RunnableParallel`, e depois uma síntese.

In [ ]:
from fabrica_agentica.chains.fase2 import chain_sequencial, chain_paralela

seq = chain_sequencial().invoke({"demanda": "Quero login com Google no Portal."})
for chave in ("resumo", "historias", "riscos"):
    print(f"--- {chave.upper()} ---\n{seq[chave]}\n")

--- RESUMO ---
Implementar autenticação com Google (login social) no Portal.

--- HISTORIAS ---
Como visitante do Portal, quero fazer login com minha conta Google, para acessar o sistema sem precisar criar e lembrar uma nova senha.
Como usuário autenticado, quero que meus dados de perfil sejam preenchidos a partir da minha conta Google, para não precisar informá-los novamente.
Como usuário, quero poder desconectar minha conta Google do Portal, para manter o controle sobre o acesso aos meus dados.

--- RISCOS ---
- **Integração e segurança OAuth:** erros na configuração do provedor Google (client ID, redirect URIs, escopos excessivos) ou validação incorreta do token podem gerar falhas de login ou brechas de segurança, como sequestro de sessão.
- **Gestão de sessão e vínculo de contas:** sem regras claras para contas já existentes com o mesmo e-mail, há risco de duplicidade de usuários, vinculação indevida de contas ou perda de sessão ao desconectar o Google.
- **Sincronização e privacid

In [ ]:
entrada = {"demanda": "Login com Google no Portal", "consulta_norma": "provedores de identidade federada login social",
           "pacote": "authlib", "projeto": "Portal"}
inicio = time.perf_counter()
par = chain_paralela().invoke(entrada)
print(f"Tempo total (3 checagens em paralelo + síntese): {time.perf_counter() - inicio:.1f}s\n")
print("CAPACIDADE:", par["capacidade"]); print("BIBLIOTECA:", par["biblioteca"]); print("NORMAS:", par["normas"])
print("\nSÍNTESE:\n", par["sintese"])

Tempo total (3 checagens em paralelo + síntese): 4.1s

CAPACIDADE: {'sprint': 13, 'dias_uteis': 9, 'capacidade_liquida': 25, 'pontos_comprometidos': 11, 'saldo_disponivel': 14}
BIBLIOTECA: {'pacote': 'authlib', 'versao_mais_recente': '1.9.5', 'vulnerabilidades': 0, 'maior_cvss': 0}
NORMAS: [{'doc': 'MAN-05', 'secao': '2. Provedores de identidade federada', 'chunk': 'CHK_025', 'similaridade': 'alta'}, {'doc': 'MAN-07', 'secao': 'ADR-012 — Provedor de identidade central', 'chunk': 'CHK_039', 'similaridade': 'moderada'}, {'doc': 'MAN-05', 'secao': 'MAN-05 — Padrão de Autenticação e Gestão de Identidade', 'chunk': 'CHK_023', 'similaridade': 'baixa'}]

SÍNTESE:
 **Viabilidade: condicionada.** A biblioteca authlib está na versão 1.9.5, com 0 vulnerabilidades e maior CVSS 0, e a sprint 13 tem saldo disponível de 14 pontos (capacidade líquida 25, pontos comprometidos 11), o que comporta a demanda.
A norma MAN-05, seção "2. Provedores de identidade federada" (CHK_025, similaridade alta), é ader

## 5. Roteamento dinâmico com `RouterChain`

Um `LLMRouterChain` lê a pergunta e escolhe, em tempo de execução, qual prompt especialista atende: **planejamento**, **consulta técnica** ou **segurança** (`MultiPromptChain`).

In [ ]:
from fabrica_agentica.chains.fase2 import criar_router_chain

roteador = criar_router_chain(verbose=True)
for pergunta in ["Quantos pontos uma história pode ter antes de precisar ser quebrada?",
                 "Posso usar pyjwt 2.8.0?",
                 "Qual a regra para fazer deploy na sexta-feira?"]:
    saida = roteador.invoke({"input": pergunta})
    print("RESPOSTA:", saida["text"][:400], "\n")



> Entering new MultiPromptChain chain...


planejamento: {'input': 'Quantos pontos uma história pode ter antes de precisar ser quebrada?'}


> Finished chain.
RESPOSTA: Uma história pode ter **até 8 pontos**. Acima disso, ela deve ser quebrada em histórias menores antes de entrar na sprint.

Na escala Fibonacci (1, 2, 3, 5, 8), 8 é o limite. Uma história de 13 pontos ou mais é considerada grande demais para planejar com segurança, pois tende a ter mais incerteza e risco de não ser concluída no ciclo. 



> Entering new MultiPromptChain chain...


seguranca: {'input': 'Posso usar pyjwt 2.8.0?'}


> Finished chain.
RESPOSTA: **Resposta curta:** Provavelmente sim, mas não use sem antes verificar na base OSV, e recomendo atualizar para a versão 2.x mais recente.

**1. MAN-04 (CVSS ≥ 7,0)**
- Até onde sei, o PyJWT 2.8.0 não tem CVE conhecida com CVSS ≥ 7,0. A vulnerabilidade séria de confusão de algoritmo (CVE-2022-29217) foi corrigida na 2.4.0, então a 2.8.0 já a inclui.
- Como minha base pode estar desatualizada, confi 



> Entering new MultiPromptChain chain...


consulta_tecnica: {'input': 'Qual a regra para fazer deploy na sexta-feira?'}


> Finished chain.
RESPOSTA: Não tenho acesso ao Manual de Engenharia da sua organização, então não posso afirmar qual é a regra exata sobre deploy na sexta-feira. Inventar uma resposta aqui poderia levar a uma decisão errada em produção.

O que posso dizer com segurança é o que costuma aparecer nesses manuais, em especial quando há janela de congelamento: normalmente existe uma política específica para sexta-feira e fim de s 



### Limite do roteamento sem ferramentas (motivação para o agente)

As rotas acima respondem só com a memória paramétrica do modelo. Compare a resposta da rota de **segurança** sobre `pyjwt 2.8.0` com o dado **real** da base OSV, consultado agora pela ferramenta. Este é o argumento central do projeto: dados que mudam com o tempo exigem ferramentas, e o roteador sozinho não basta.

In [9]:
osv = json.loads(consultar_vulnerabilidades.invoke({"pacote": "pyjwt", "versao": "2.8.0"}))
print(f"OSV (real): pyjwt 2.8.0 tem {osv['total']} vulnerabilidades conhecidas; maior CVSS = {osv['maior_cvss']}")
for v in osv["vulnerabilidades"][:3]:
    print(f"  - {v['id']} CVSS {v['cvss']} corrigida em {v['corrigida_em']}: {v['resumo'][:90]}")

OSV (real): pyjwt 2.8.0 tem 28 vulnerabilidades conhecidas; maior CVSS = 9.1
  - GHSA-ffc3-869f-jxw9 CVSS 9.1 corrigida em ['2.14.0']: PyJWT: Asymmetric-PEM detection bypass: whitespace/line-ending-mutated public keys skip th
  - PYSEC-2026-4145 CVSS 9.1 corrigida em ['2.14.0']: PyJWT: Asymmetric-PEM detection bypass: whitespace/line-ending-mutated public keys skip th
  - PYSEC-2026-4183 CVSS 8.2 corrigida em ['2.15.0']: 


## 6. Prompt avançado: PTCF + *few-shot* + *chain-of-thought*

O prompt de sistema do agente segue o framework **PTCF** — **P**ersona, **T**arefa, **C**ontexto (com as regras de raciocínio) e **F**ormato. Três exemplos *few-shot* (inseridos no system prompt como texto rotulado "ilustrativo" — como pares de mensagens, o modelo os confundia com turnos reais, como mostra a Fase 3) ensinam o raciocínio nos casos ambíguos do domínio (biblioteca com CVE, capacidade da sprint, assunto fora do manual), e a instrução de *chain-of-thought* pede que o agente planeje antes de usar ferramentas — o que deixa o raciocínio explícito e depurável na trajetória.

In [10]:
from fabrica_agentica.prompts import SYSTEM_PTCF, EXEMPLOS_FEW_SHOT
print(SYSTEM_PTCF)
print("\n--- FEW-SHOT ---")
for e in EXEMPLOS_FEW_SHOT:
    print(f"Usuário: {e['pergunta']}\n{e['resposta']}\n")

PERSONA: Você é o Tech Lead Agêntico da NovaForja Software (empresa fictícia), o primeiro agente de uma fábrica agêntica de desenvolvimento. Você é um líder técnico sênior, rigoroso com normas e números.

TAREFA: transformar demandas em planos de trabalho verificados (épico -> histórias com critérios Gherkin -> tarefas), responder dúvidas sobre o backlog, as normas do Manual de Engenharia e as bibliotecas de software, e propor o registro do trabalho planejado.

CONTEXTO: projetos Portal, PagaFacil e AppCampo. Fontes de verdade, sempre via ferramentas: backlog (SQL), Manual de Engenharia MAN-01 a MAN-07 (busca semântica), PyPI e OSV (bibliotecas e vulnerabilidades), GitHub (issues) e BrasilAPI (feriados). Regras de raciocínio:
1. Nunca invente números, versões ou regras: capacidade de sprint SEMPRE por calcular_capacidade_sprint; versões e vulnerabilidades SEMPRE por consultar_pypi/consultar_vulnerabilidades.
2. Para normas, use buscar_manual_engenharia reescrevendo a pergunta no vocabu

## 7. Agente com *function calling*: `create_openai_tools_agent` + `AgentExecutor`

O mesmo prompt PTCF, agora com todas as ferramentas do catálogo. Com Claude, o adaptador `ChatClaudeCompat` (em `config.py`) converte as ferramentas do formato OpenAI para o formato da Anthropic, o que permite usar exatamente o construto pedido. Um *callback* de log registra cada decisão.

In [11]:
from fabrica_agentica.agentes.executor import criar_executor, trajetoria
from fabrica_agentica.observabilidade.callbacks import RegistroExecucao
from fabrica_agentica.ferramentas.catalogo import TIPOS

log = RegistroExecucao(sessao="fase2", imprimir=True)
executor = criar_executor()
inicio = time.perf_counter()
resultado = executor.invoke({"input": "Quero login com Google no projeto Portal na próxima sprint."}, config={"callbacks": [log]})
print(f"\nTempo: {time.perf_counter() - inicio:.1f}s\n\nTRAJETÓRIA:")
for passo in trajetoria(resultado):
    print(f" - [{TIPOS[passo['ferramenta']]}] {passo['ferramenta']}({json.dumps(passo['argumentos'], ensure_ascii=False)[:110]})")

  [log] llm_inicio   claude-sonnet-5-5 


  [log] llm_fim       4.28s 
  [log] decisao      calcular_capacidade_sprint {'projeto': 'Portal'}
  [log] tool_inicio  calcular_capacidade_sprint {'projeto': 'Portal'}


  [log] tool_fim     calcular_capacidade_sprint 0.44s {"dias_uteis": 9, "capacidade_bruta": 31.25, "reserva_defeitos": 6.25, "capacidade_liquida": 25, "po
  [log] decisao      buscar_manual_engenharia {'consulta_otimizada': 'autenticação federada OAuth 2.0 OpenID Connect provedor de identidade extern
  [log] tool_inicio  buscar_manual_engenharia {'consulta_otimizada': 'autenticação federada OAuth 2.0 OpenID Connect provedor de identidade extern
  [log] tool_fim     buscar_manual_engenharia 0.04s {"consulta": "autenticação federada OAuth 2.0 OpenID Connect provedor de identidade externo, fluxo d
  [log] decisao      buscar_manual_engenharia {'consulta_otimizada': 'segurança de dependências, adoção de biblioteca, severidade CVSS, defasagem 
  [log] tool_inicio  buscar_manual_engenharia {'consulta_otimizada': 'segurança de dependências, adoção de biblioteca, severidade CVSS, defasagem 
  [log] tool_fim     buscar_manual_engenharia 0.01s {"consulta": "segurança de dependências, adoção de b

  [log] tool_fim     buscar_issues_github 0.46s {"repositorio": "DriccoC/fabrica-agentica-sandbox", "modo": "ao vivo", "issues": []}
  [log] decisao      buscar_issues_github {'termo': 'Google'}
  [log] tool_inicio  buscar_issues_github {'termo': 'Google'}


  [log] tool_fim     buscar_issues_github 0.47s {"repositorio": "DriccoC/fabrica-agentica-sandbox", "modo": "ao vivo", "issues": []}
  [log] llm_inicio   claude-sonnet-5-5 


  [log] llm_fim       2.97s 
  [log] decisao      consultar_vulnerabilidades {'pacote': 'authlib', 'versao': '1.9.5'}
  [log] tool_inicio  consultar_vulnerabilidades {'pacote': 'authlib', 'versao': '1.9.5'}


  [log] tool_fim     consultar_vulnerabilidades 0.71s {"pacote": "authlib", "versao": "1.9.5", "total": 0, "vulnerabilidades": [], "maior_cvss": 0}
  [log] decisao      consultar_backlog_sql {'consulta_sql': "SELECT nome, descricao, stack FROM projetos WHERE nome='Portal'"}
  [log] tool_inicio  consultar_backlog_sql {'consulta_sql': "SELECT nome, descricao, stack FROM projetos WHERE nome='Portal'"}
[auditoria SQL] SELECT nome, descricao, stack FROM projetos WHERE nome='Portal' LIMIT 50
  [log] tool_fim     consultar_backlog_sql 0.0s {"ok": true, "sql": "SELECT nome, descricao, stack FROM projetos WHERE nome='Portal' LIMIT 50", "tot
  [log] llm_inicio   claude-sonnet-5-5 


  [log] llm_fim       3.38s 
  [log] decisao      decompor_demanda {'demanda': 'Login com Google no Portal (portal do cliente web, FastAPI + React) na sprint 13', 'con
  [log] tool_inicio  decompor_demanda {'demanda': 'Login com Google no Portal (portal do cliente web, FastAPI + React) na sprint 13', 'con
  [log] llm_inicio   claude-haiku-5-5 


  [log] llm_fim       7.68s 
  [log] tool_fim     decompor_demanda 7.69s {"plano": {"epico": "Login com Google no Portal (PF) via Keycloak", "objetivo": "Permitir que pessoa
  [log] llm_inicio   claude-sonnet-5-5 


  [log] llm_fim       8.19s 
  [log] resposta      [{'text': '**Conclusão:** o login com Google cabe na próxima sprint, a sprint 13 do Portal. O plano 

Tempo: 28.8s

TRAJETÓRIA:
 - [hardcoded] calcular_capacidade_sprint({"projeto": "Portal"})
 - [RAG (FAISS)] buscar_manual_engenharia({"consulta_otimizada": "autenticação federada OAuth 2.0 OpenID Connect provedor de identidade externo, fluxo d)
 - [RAG (FAISS)] buscar_manual_engenharia({"consulta_otimizada": "segurança de dependências, adoção de biblioteca, severidade CVSS, defasagem de versão )
 - [API externa] consultar_pypi({"pacote": "authlib"})
 - [API externa] buscar_issues_github({"termo": "login"})
 - [API externa] buscar_issues_github({"termo": "Google"})
 - [API externa] consultar_vulnerabilidades({"pacote": "authlib", "versao": "1.9.5"})
 - [text-to-query] consultar_backlog_sql({"consulta_sql": "SELECT nome, descricao, stack FROM projetos WHERE nome='Portal'"})
 - [semântica] decompor_demanda({"demanda": "Login com Google no

In [12]:
print(texto_de(resultado["output"]))

**Conclusão:** o login com Google cabe na próxima sprint, a sprint 13 do Portal. O plano tem 11 pontos e o saldo é de 14, então sobram 3. O Google deve entrar como provedor federado no Keycloak, não direto no Portal. Ainda não registrei nada no backlog.

**Fundamentos**
- **Capacidade (calcular_capacidade_sprint, MAN-06 §3):**
  - A sprint 13 vai de 09/11 a 20/11/2026 e tem 9 dias úteis. O dia 20/11 é feriado (BrasilAPI).
  - A capacidade líquida é de 25 pontos, com 11 já comprometidos, o que deixa saldo de 14.
- **Arquitetura (MAN-07 ADR-012, CHK_039):** o Keycloak é o provedor de identidade central. O Google é configurado como provedor federado nele, e o Portal se integra só ao Keycloak via OIDC.
- **Protocolo (MAN-05 §1, CHK_024):** OIDC com fluxo de código de autorização e PKCE. Os fluxos implícito e de senha são proibidos.
- **Público (MAN-05 §2, CHK_025):**
  - O login social vale só para pessoa física.
  - Exige vinculação a uma conta interna e verificação do e-mail.
  - Cliente

## 8. Componentes de confiabilidade

### 8.1 *Output parsers* com tratamento de erro

Três camadas: (a) **saída estruturada nativa** (`with_structured_output(..., method="json_schema")`), que restringe o modelo ao esquema — usada nas ferramentas semânticas; (b) **`OutputFixingParser`**, que devolve um texto malformado a um LLM para corrigi-lo no esquema; (c) **`handle_parsing_errors=True`** no `AgentExecutor`, que transforma um erro de formato em observação para o agente tentar de novo, em vez de derrubar a execução.

In [13]:
from langchain_classic.output_parsers import OutputFixingParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.exceptions import OutputParserException
from fabrica_agentica.esquemas import IntencaoEntidades

parser = PydanticOutputParser(pydantic_object=IntencaoEntidades)
texto_malformado = "intencao: checar_dependencia; pacote authlib, versão 1.3.0, projeto Portal"
try:
    parser.parse(texto_malformado)
except OutputParserException as erro:
    print("Parser simples falhou:", str(erro)[:120], "...")

corretor = OutputFixingParser.from_llm(parser=parser, llm=criar_llm("auxiliar"))
print("OutputFixingParser corrigiu:", corretor.parse(texto_malformado))
print("Executor com handle_parsing_errors =", executor.handle_parsing_errors)

Parser simples falhou: Invalid json output: intencao: checar_dependencia; pacote authlib, versão 1.3.0, projeto Portal
For troubleshooting, vis ...


OutputFixingParser corrigiu: intencao='checar_dependencia' projeto='Portal' sprint=None pacote='authlib' versao='1.3.0' prioridade=None pessoa=None
Executor com handle_parsing_errors = True


### 8.2 Memória da chain: buffer × resumo × limite de tokens

Comparamos três estratégias com a mesma conversa de planejamento.

In [14]:
from langchain_classic.memory import ConversationBufferMemory, ConversationSummaryMemory, ConversationTokenBufferMemory
from fabrica_agentica.prompts import PROMPT_RESUMO_MEMORIA

conversa = [
    ("Vamos planejar a sprint 13 do Portal.", "Certo: Portal, sprint 13, de 09/11 a 20/11, com 14 pontos de saldo."),
    ("A demanda é login com Google para pessoa física.", "Pelo MAN-05, login social só para pessoa física, com OIDC e PKCE."),
    ("Use a authlib.", "A authlib 1.9.5 é a mais recente e não tem vulnerabilidades conhecidas."),
    ("E se tirarmos o Bruno da sprint?", "Sem o Bruno, a capacidade líquida cai de 25 para 20 e o saldo para 9 pontos."),
]
memorias = {
    "buffer (tudo)": ConversationBufferMemory(),
    "resumo (LLM)": ConversationSummaryMemory(llm=criar_llm("auxiliar"), prompt=PROMPT_RESUMO_MEMORIA),
    "limite de 120 tokens": ConversationTokenBufferMemory(llm=criar_llm("auxiliar"), max_token_limit=120),
}
for nome, memoria in memorias.items():
    for entrada_usuario, saida_ia in conversa:
        memoria.save_context({"input": entrada_usuario}, {"output": saida_ia})
    conteudo = memoria.load_memory_variables({})["history"]
    print(f"=== {nome}: {len(conteudo)} caracteres ===\n{conteudo}\n")

=== buffer (tudo): 461 caracteres ===
Human: Vamos planejar a sprint 13 do Portal.
AI: Certo: Portal, sprint 13, de 09/11 a 20/11, com 14 pontos de saldo.
Human: A demanda é login com Google para pessoa física.
AI: Pelo MAN-05, login social só para pessoa física, com OIDC e PKCE.
Human: Use a authlib.
AI: A authlib 1.9.5 é a mais recente e não tem vulnerabilidades conhecidas.
Human: E se tirarmos o Bruno da sprint?
AI: Sem o Bruno, a capacidade líquida cai de 25 para 20 e o saldo para 9 pontos.



/var/folders/gn/y128ffbn6790ggg9s75xk7w80000gn/T/ipykernel_70831/1643172940.py:12: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  "resumo (LLM)": ConversationSummaryMemory(llm=criar_llm("auxiliar"), prompt=PROMPT_RESUMO_MEMORIA),
/var/folders/gn/y128ffbn6790ggg9s75xk7w80000gn/T/ipykernel_70831/1643172940.py:13: LangChainDeprecationWarning: The class `ConversationTokenBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://doc

=== resumo (LLM): 568 caracteres ===
A sessão de planejamento técnico trata da sprint 13 do Portal (09/11 a 20/11), cuja demanda é o login com Google para pessoa física, a única modalidade de login social permitida pelo MAN-05, usando OIDC e PKCE. Decidiu-se usar a biblioteca authlib, indicada na versão 1.9.5 como a mais recente e sem vulnerabilidades conhecidas, informação ainda não verificada. Bruno foi retirado da sprint, o que reduz a capacidade líquida de 25 para 20 pontos e o saldo disponível de 14 para 9 pontos. Como pendência, falta verificar a versão e as vulnerabilidades da authlib 1.9.5.



=== limite de 120 tokens: 218 caracteres ===
Human: Use a authlib.
AI: A authlib 1.9.5 é a mais recente e não tem vulnerabilidades conhecidas.
Human: E se tirarmos o Bruno da sprint?
AI: Sem o Bruno, a capacidade líquida cai de 25 para 20 e o saldo para 9 pontos.



**Escolha para o agente:** o volume esperado é de sessões de planejamento com 5 a 20 turnos, em que detalhes antigos importam (projeto, sprint, pessoas retiradas) — o limite de tokens perderia "Portal, sprint 13" (veja acima); o buffer completo cresce sem limite e encarece cada chamada (o agente já consome ~40 mil tokens de entrada numa demanda complexa). Adotamos **janela deslizante + resumo** (buffer dos turnos recentes e resumo dos antigos), complementada na Fase 3 por memória de longo prazo com busca vetorial.

### 8.3 Busca semântica com FAISS como ferramenta

O agente já usa `buscar_manual_engenharia` (índice FAISS do Manual de Engenharia) na trajetória da seção 7. Cada resultado traz norma, seção, `chunk_id` e diagnóstico de similaridade:

In [15]:
from fabrica_agentica.ferramentas.catalogo import buscar_manual_engenharia
saida = json.loads(buscar_manual_engenharia.invoke({"consulta_otimizada": "provedores de identidade federada login social Google"}))
for r in saida["resultados"]:
    print(f"{r['distancia_l2']:.3f} [{r['similaridade']:8}] {r['chunk_id']} {r['doc_id']} | {r['secao']}")

0.503 [alta    ] CHK_025 MAN-05 | 2. Provedores de identidade federada
0.775 [alta    ] CHK_039 MAN-07 | ADR-012 — Provedor de identidade central
1.269 [baixa   ] CHK_041 MAN-07 | ADR-017 — Banco de dados por serviço
1.277 [baixa   ] CHK_028 MAN-05 | 5. Autenticação multifator


### 8.4 *Callbacks* para logging e depuração

O `RegistroExecucao` (em `observabilidade/callbacks.py`) gravou cada evento da execução da seção 7 em `logs/execucoes.jsonl`. O resumo mostra custo e onde o tempo foi gasto — útil para depurar (ex.: uma ferramenta lenta) e para estimar custo por demanda.

In [16]:
resumo = log.resumo()
print(json.dumps({k: v for k, v in resumo.items() if k != "erros"}, ensure_ascii=False, indent=1))
print("erros:", len(resumo["erros"]))
print("\nPrimeiros eventos do log JSONL:")
for e in log.eventos[:6]:
    print(" ", json.dumps(e, ensure_ascii=False)[:160])

{
 "chamadas_llm": 5,
 "tokens_entrada": 35011,
 "tokens_saida": 3740,
 "ferramentas": [
  "calcular_capacidade_sprint",
  "buscar_manual_engenharia",
  "buscar_manual_engenharia",
  "consultar_pypi",
  "buscar_issues_github",
  "buscar_issues_github",
  "consultar_vulnerabilidades",
  "consultar_backlog_sql",
  "decompor_demanda"
 ],
 "tempo_llm_s": 26.5,
 "tempo_ferramentas_s": 9.9
}
erros: 0

Primeiros eventos do log JSONL:
  {"ts": "2026-11-05T17:09:43.319", "sessao": "fase2", "tipo": "llm_inicio", "modelo": "claude-sonnet-5-5", "n_mensagens": 2}
  {"ts": "2026-11-05T17:09:47.597", "sessao": "fase2", "tipo": "llm_fim", "duracao_s": 4.28, "tokens": {"input_tokens": 5170, "output_tokens": 483, "total_tokens"
  {"ts": "2026-11-05T17:09:47.601", "sessao": "fase2", "tipo": "decisao", "ferramenta": "calcular_capacidade_sprint", "entrada": "{'projeto': 'Portal'}"}
  {"ts": "2026-11-05T17:09:47.601", "sessao": "fase2", "tipo": "tool_inicio", "ferramenta": "calcular_capacidade_sprint", "ent

### 8.5 Justificativa das escolhas de confiabilidade

| Componente | Escolha | Por quê |
|---|---|---|
| Saída estruturada | `json_schema` nativo nas ferramentas semânticas; `PydanticOutputParser` na chain MVP | o esquema nativo restringe a geração e elimina a maior parte dos erros de formato; o parser explícito mantém o MVP portável entre provedores |
| Tratamento de erro | `OutputFixingParser` + `handle_parsing_errors` + nova tentativa no `@chain` | um JSON quebrado ou um passo fora do formato não deve derrubar uma sessão de planejamento; o custo de uma chamada extra é menor que o de reiniciar |
| Memória | janela + resumo (curto prazo) e, na Fase 3, longo prazo vetorial | preserva fatos antigos e decisivos da sessão sem crescer sem limite |
| Busca semântica | FAISS local com embeddings multilíngues gratuitos | o manual é pequeno e estável; FAISS é rápido, sem servidor, e o índice é reaproveitado; o diagnóstico de distância permite detectar respostas sem cobertura |
| Callbacks | log JSONL local + LangSmith | o log local funciona sem conta nem rede (avaliação do professor) e alimenta a interface; o LangSmith dá a árvore de execução e as métricas de avaliação |